# 竞赛主 Notebook（jingsai 环境，CPU 一次跑通）

## 记忆口诀
**读 params** → 洗 → 特 → 树×3 → stack_v1 → RNN(Keras) → stack_v2

## 调参（本机 dinov3）
`tune.ipynb` + PyTorch GPU → 产出 `params.json` → 拷到本目录

## 顺序
B清洗 → A debug树 → C全特征+A重训树 → stack_v1 → RNN → stack_v2

In [2]:
import sys; sys.path.insert(0, '.')
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from lib_clean import load
from lib_feat import build, TIME, TARGET
from lib_model import split, train_tree, compare, stack, train_rnn, predict_rnn_test, load_params, mape

params = load_params()
PATH_TRAIN = '../data/实验数据1.xlsx'
PATH_TEST  = '../data/data_test1.xlsx'
PATH_FB    = '../data/cleaned_df.csv'
VAL_START  = params['val_start']
TREES      = params['trees']
RNN_CFG    = params['rnn']

In [3]:
# [B] 清洗
df_hist, df_test = load(PATH_TRAIN, PATH_TEST, PATH_FB)
df_all = pd.concat([df_hist, df_test]).sort_values(TIME).reset_index(drop=True)
print(len(df_hist), len(df_test))

116734 2880


c:\Users\zhich\OneDrive\Desktop\github\竞赛\202608调度\comp\.\lib_clean.py:30: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  return df.interpolate('time').ffill().bfill().reset_index(names=TIME)
c:\Users\zhich\OneDrive\Desktop\github\竞赛\202608调度\comp\.\lib_clean.py:30: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  return df.interpolate('time').ffill().bfill().reset_index(names=TIME)


In [4]:
# [A] Stage1 debug：7维原始特征，先跑通树
df1, feats1 = build(df_all, full=False)
ds1 = split(df1, VAL_START, feats1)
Xtr, ytr, _ = ds1['train']; Xva, yva, _ = ds1['val']; Xte, _, dte = ds1['test']
res1 = {}
for name in TREES:
    m = train_tree(name, Xtr, ytr, Xva, yva)
    res1[name] = {'y': yva, 'p': m.predict(Xva), 'test': m.predict(Xte)}
compare(res1)

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[7]	valid_0's l2: 6.23993e+06
model     mape%
  xgb 10.187441
  cat 10.293964
  lgb 10.322214


,model,mape%
0,lgb,10.322214
1,xgb,10.187441
2,cat,10.293964


In [5]:
# [C→A] Stage2：30维全特征，重训树
df2, feats2 = build(df_all, full=True)
ds2 = split(df2, VAL_START, feats2)
Xtr, ytr, _ = ds2['train']; Xva, yva, _ = ds2['val']; Xte, _, dte = ds2['test']
res2 = {}
for name in TREES:
    m = train_tree(name, Xtr, ytr, Xva, yva)
    res2[name] = {'y': yva, 'p': m.predict(Xva), 'test': m.predict(Xte)}
compare(res2)

Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[473]	valid_0's l2: 617579
model    mape%
  lgb 2.593726
  cat 2.595529
  xgb 2.604831


,model,mape%
0,lgb,2.593726
1,xgb,2.604831
2,cat,2.595529


In [6]:
# [B] Stacking v1：仅树
tree_val  = {k: v['p'] for k, v in res2.items()}
tree_test = {k: v['test'] for k, v in res2.items()}
pva1, pte1, _ = stack(tree_val, yva, tree_test, alpha=params['stack_alpha'])
print(f'[stack_v1] mape={mape(yva, pva1)*100:.2f}%')
pd.DataFrame({TIME: dte, TARGET: pte1}).to_csv('final_trees.csv', index=False)

[stack_v1] mape=2.35%


In [7]:
# [C] RNN — 参数来自 params.json
ht = pd.concat([df2[df2['_src']=='hist'], df2[df2['_src']=='test']]).reset_index(drop=True)
n_hist, n_tr = (df2['_src']=='hist').sum(), len(ds2['train'][0])
hist = ht['_src'] == 'hist'
sx = StandardScaler().fit(ht.loc[hist, feats2].values)
sy = StandardScaler().fit(ht.loc[hist, [TARGET]].values)
X = sx.transform(ht[feats2].values).astype(np.float32)
y = ht[TARGET].ffill().fillna(0).values
seq = RNN_CFG['seq']
i0 = max(0, n_tr - seq)

res_rnn = {}
for kind in params['rnn_models']:
    m, pva = train_rnn(kind, X[:n_tr], y[:n_tr], X[i0:n_hist], y[i0:n_hist], sy, params=RNN_CFG)
    res_rnn[kind] = {'p': pva[-len(yva):], 'test': predict_rnn_test(m, X, y, seq, n_hist, len(Xte), sy)}
    print(f'{kind} mape={mape(yva, pva[-len(yva):])*100:.2f}%')




gru mape=1.34%


In [8]:
# [B] Stacking v2：树 + RNN
all_val  = {**tree_val,  **{k: v['p'] for k, v in res_rnn.items()}}
all_test = {**tree_test, **{k: v['test'] for k, v in res_rnn.items()}}
# [B] Stacking v2：树 + RNN
all_val  = {**tree_val,  **{k: v['p'] for k, v in res_rnn.items()}}
all_test = {**tree_test, **{k: v['test'] for k, v in res_rnn.items()}}
pva2, pte2, _ = stack(all_val, yva, all_test, alpha=params['stack_alpha'])
print(f'[stack_v2] mape={mape(yva, pva2)*100:.2f}%')
pd.DataFrame({TIME: dte, TARGET: pte2}).to_csv('final_stack.csv', index=False)

[stack_v2] mape=1.30%
